# 🌲 MÓDULO 3: Entrenamiento de Modelos
## Árbol de Decisión vs Random Forest

**Objetivo:** Entrenar ambos modelos y comparar su rendimiento.

---

## 3.1 Configuración y Preparación de Datos

In [1]:
import sys
import time

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    print("📍 Ejecutando en Google Colab")
    RUTA_DATOS = '/content/drive/MyDrive/ML_Vuelos/data/'
else:
    print("🐳 Ejecutando en Docker")
    RUTA_DATOS = '../data/'

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Vuelos_Modulo3") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

🐳 Ejecutando en Docker


In [2]:
# Cargar y preparar datos (repetimos del Módulo 2)
from pyspark.ml.feature import StringIndexer, VectorAssembler

df = spark.read.csv(RUTA_DATOS + "flights_2015_full.csv", header=True, inferSchema=True)

# Indexar categóricas
for col_name in ["AIRLINE", "ORIGIN", "DEST"]:
    indexer = StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    df = indexer.fit(df).transform(df)

# Crear vector de features
feature_cols = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "AIRLINE_idx", "ORIGIN_idx", "DEST_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df).select("features", "DEP_DEL15")

# Dividir
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"✅ Datos preparados: {train_df.count():,} train / {test_df.count():,} test")

✅ Datos preparados: 4,265,954 train / 1,066,960 test


## 3.2 Entrenar Árbol de Decisión 🌳

In [3]:
from pyspark.ml.classification import DecisionTreeClassifier

print("🌳 Entrenando Árbol de Decisión...")
inicio = time.time()

dt = DecisionTreeClassifier(
    labelCol="DEP_DEL15",
    featuresCol="features",
    maxDepth=8,
    minInstancesPerNode=100,
    maxBins=330           # ← esto es lo que se agrega
)

modelo_arbol = dt.fit(train_df)
tiempo_arbol = time.time() - inicio

print(f"✅ Árbol entrenado en {tiempo_arbol:.2f} segundos")
print(f"   Profundidad: {modelo_arbol.depth}")
print(f"   Nodos: {modelo_arbol.numNodes}")

🌳 Entrenando Árbol de Decisión...
✅ Árbol entrenado en 70.43 segundos
   Profundidad: 8
   Nodos: 17


In [4]:
# Feature Importance del Árbol
print("\n📊 IMPORTANCIA DE VARIABLES (Árbol)")
print("="*40)
importancias = modelo_arbol.featureImportances.toArray()
for nombre, imp in sorted(zip(feature_cols, importancias), key=lambda x: -x[1]):
    barra = "█" * int(imp * 50)
    print(f"{nombre:15} {imp:.4f} {barra}")


📊 IMPORTANCIA DE VARIABLES (Árbol)
DEP_HOUR        0.8057 ████████████████████████████████████████
AIRLINE_idx     0.1329 ██████
MONTH           0.0404 ██
ORIGIN_idx      0.0211 █
DAY_OF_WEEK     0.0000 
DISTANCE        0.0000 
DEST_idx        0.0000 


In [5]:
# Ver estructura del árbol
print("\n🌳 ESTRUCTURA DEL ÁRBOL (primeras líneas):")
print(modelo_arbol.toDebugString[:2000])


🌳 ESTRUCTURA DEL ÁRBOL (primeras líneas):
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_c6bd1abab774, depth=8, numNodes=17, numClasses=2, numFeatures=7
  If (feature 2 <= 11.5)
   Predict: 0.0
  Else (feature 2 > 11.5)
   If (feature 4 in {1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0})
    Predict: 0.0
   Else (feature 4 not in {1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0})
    If (feature 2 <= 15.5)
     Predict: 0.0
    Else (feature 2 > 15.5)
     If (feature 5 in {5.0,9.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,21.0,22.0,25.0,26.0,29.0,30.0,34.0,35.0,36.0,37.0,38.0,39.0,40.0,41.0,42.0,43.0,44.0,45.0,46.0,47.0,48.0,49.0,50.0,53.0,54.0,55.0,56.0,57.0,58.0,59.0,60.0,62.0,63.0,64.0,65.0,66.0,67.0,68.0,69.0,70.0,72.0,73.0,75.0,76.0,77.0,78.0,83.0,84.0,85.0,87.0,88.0,89.0,98.0,99.0,100.0,102.0,114.0,118.0,119.0,121.0,124.0,126.0,128.0,149.0})
      Predict: 0.0
     Else (feature 5 not in {5.0,9.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,21.0,22.0,25.0,26.0,29.

## 3.3 Entrenar Random Forest 🌲🌲🌲

In [6]:
from pyspark.ml.classification import RandomForestClassifier

print("🌲 Entrenando Random Forest (100 árboles)...")
inicio = time.time()

rf = RandomForestClassifier(
    labelCol="DEP_DEL15",
    featuresCol="features",
    numTrees=80,        # Número de árboles
    maxDepth=8,          # Profundidad máxima por árbol
    seed=42,
    maxBins=330
)

modelo_bosque = rf.fit(train_df)
tiempo_bosque = time.time() - inicio

print(f"✅ Bosque entrenado en {tiempo_bosque:.2f} segundos")
print(f"   Árboles: {modelo_bosque.numTrees}")

🌲 Entrenando Random Forest (100 árboles)...
✅ Bosque entrenado en 821.92 segundos
   Árboles: RandomForestClassifier_580e9127af14__numTrees


In [7]:
# Feature Importance del Bosque
print("\n📊 IMPORTANCIA DE VARIABLES (Random Forest)")
print("="*40)
importancias_rf = modelo_bosque.featureImportances.toArray()
for nombre, imp in sorted(zip(feature_cols, importancias_rf), key=lambda x: -x[1]):
    barra = "█" * int(imp * 50)
    print(f"{nombre:15} {imp:.4f} {barra}")


📊 IMPORTANCIA DE VARIABLES (Random Forest)
DEP_HOUR        0.6583 ████████████████████████████████
AIRLINE_idx     0.1704 ████████
ORIGIN_idx      0.0878 ████
MONTH           0.0563 ██
DEST_idx        0.0129 
DISTANCE        0.0113 
DAY_OF_WEEK     0.0030 


## 3.4 Comparación de Tiempos

In [8]:
print("\n⏱️ COMPARACIÓN DE TIEMPOS DE ENTRENAMIENTO")
print("="*45)
print(f"{'Modelo':<25} {'Tiempo':>10} {'Factor':>8}")
print("-"*45)
print(f"{'Árbol de Decisión':<25} {tiempo_arbol:>8.2f}s {1.0:>7.1f}x")
print(f"{'Random Forest (100)':<25} {tiempo_bosque:>8.2f}s {tiempo_bosque/tiempo_arbol:>7.1f}x")
print("="*45)


⏱️ COMPARACIÓN DE TIEMPOS DE ENTRENAMIENTO
Modelo                        Tiempo   Factor
---------------------------------------------
Árbol de Decisión            70.43s     1.0x
Random Forest (100)         821.92s    11.7x


---
## ✅ CHECKPOINT MÓDULO 3

Ahora tienes:

| Modelo | Variable | Tiempo |
|--------|----------|--------|
| Árbol de Decisión | `modelo_arbol` | Rápido |
| Random Forest | `modelo_bosque` | Más lento |

**Siguiente:** → `Modulo4_Evaluacion.ipynb`